# CDEaDPP 2026.2 — Trabalho Final em Python

## 1. Delimitação do problema e desenho da pesquisa

Este notebook é um **template de protocolo replicável**, ilustrado por dados
inteiramente **simulados**. O caso hipotético avalia pedidos de benefício
social registrados entre 2023 e 2025 em unidades administrativas. A exposição
é o atendimento por um programa de orientação jurídica; a participação não
resulta de sorteio. Cada linha representa um pedido, não uma pessoa nem uma
unidade. Na adaptação a dados reais, o estudante deverá verificar se pedidos
repetidos da mesma pessoa exigem outro nível de agrupamento.

**Problema delimitado (tempo, espaço, matéria e população):** entre pedidos
elegíveis protocolados nas unidades participantes no período definido, qual é
a associação entre atendimento pelo programa e pontuação de acesso efetivo ao
benefício, considerando características do caso medidas antes do atendimento?
Como análise secundária, a distribuição de cinco desfechos administrativos
difere entre atendidos e não atendidos?

| Elemento | Protocolo didático | Decisão a registrar ao adaptar o estudo |
|---|---|---|
| População-alvo | Pedidos elegíveis no período e território hipotéticos | Definir destinatários da inferência e fonte da lista completa |
| População acessível | Pedidos presentes no sistema administrativo simulado | Explicar cobertura, acesso e exclusões |
| Unidade de análise | Pedido administrativo | Verificar pedidos múltiplos por requerente |
| Exposição | Atendimento pelo programa, registrado como Sim/Não | Fixar momento de início e critérios de elegibilidade |
| Desfecho primário | Pontuação de acesso efetivo, em escala intervalar assumida | Descrever itens, pesos e evidências de validade |
| Desfecho secundário | Um entre cinco estados administrativos nominais | Fixar janela de observação e regras de codificação |
| Inclusão | Pedido elegível com registro no intervalo definido | Publicar critérios antes de olhar os resultados |
| Exclusão | Nenhuma exclusão silenciosa na simulação | Registrar perdas, duplicatas e motivos de exclusão |

**Hipótese primária, bilateral:** $H_0: \beta_{atendimento}=0$ versus
$H_1: \beta_{atendimento}\ne 0$ no modelo previamente especificado, com
$\alpha=0{,}05$. A hipótese secundária afirma que as distribuições dos cinco
desfechos são iguais entre grupos ($H_0$), contra diferença em pelo menos uma
categoria ($H_1$). O poder planejado é 85% para um efeito mínimo que deverá
ser justificado substantivamente, não escolhido depois de observar o p-valor.

**Caminhos plausíveis:** complexidade prévia, vulnerabilidade e unidade podem
afetar a chance de atendimento e o resultado. Acesso digital ou qualidade da
orientação anterior podem não constar da base. O caminho
`acesso digital não observado → atendimento` e
`acesso digital não observado → resultado` exemplifica confundimento residual.
Um ajuste pelas variáveis observadas não bloqueia necessariamente esse caminho.

**Para preencher com dados reais:** explicite o fundamento teórico do efeito,
uma explicação rival, o sistema de referência, o período, as unidades, a forma
de seleção, as medidas de cada constructo e a direção plausível dos vieses.
Diferencie validade interna (comparabilidade), validade externa (população
alcançada) e validade de mensuração (o que o instrumento realmente mede).

In [ ]:
# Importar bibliotecas numéricas, de tabelas, gráficos e inferência.
import importlib.metadata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
from IPython.display import display
from statsmodels.stats.power import GofChisquarePower, TTestIndPower

# Definir parâmetros antes de gerar e examinar os dados.
SEMENTE = 2026
alpha = 0.05
beta = 0.15
poder_alvo = 0.85
rng = np.random.default_rng(SEMENTE)

# Localizar a raiz por seus arquivos de referência, sem fixar um caminho local.
candidatos = (Path.cwd(), *Path.cwd().parents)
raizes = [
    pasta
    for pasta in candidatos
    if (pasta / "pyproject.toml").is_file()
    and (pasta / "CDEaDPP_2026_Trabalho_Final_Python.ipynb").is_file()
]
if not raizes:
    raise FileNotFoundError(
        "Abra o notebook na raiz do projeto ou em uma subpasta dele."
    )
RAIZ_PROJETO = raizes[0].resolve()
PASTA_SAIDA = (RAIZ_PROJETO / "out" / "cdedpp_2026_v2").resolve()

# Garantir que os arquivos produzidos fiquem dentro do projeto.
assert isinstance(SEMENTE, int), "A semente deve ser um inteiro."
assert 0 < alpha < 1, "O erro tipo I deve estar entre zero e um."
assert 0 < beta < 1, "O erro tipo II deve estar entre zero e um."
assert np.isclose(poder_alvo, 1 - beta), "Poder deve ser 1 − beta."
assert PASTA_SAIDA.is_relative_to(RAIZ_PROJETO), "Saída fora da raiz."
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# Preparar o estilo visual e a precisão das tabelas para as próximas seções.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.precision", 3)

# Mostrar a semente, os limites de decisão e as versões que afetam a replicação.
pacotes = ("numpy", "pandas", "matplotlib", "seaborn", "scipy", "statsmodels")
versoes = {nome: importlib.metadata.version(nome) for nome in pacotes}
print("Raiz do projeto:", RAIZ_PROJETO)
print("Semente:", SEMENTE, "| alfa:", alpha, "| beta:", beta)
print("Poder planejado:", poder_alvo)
display(pd.DataFrame(versoes.items(), columns=["Pacote", "Versão"]))

### Interpretação do desenho

A semente torna reproduzíveis as amostras **simuladas**; ela não randomiza a
entrada no programa. Portanto, diferenças posteriores podem refletir o
atendimento, a seleção de casos, características omitidas ou erro de medida.
A análise ajustada deverá ser descrita como associação condicional. Mesmo que
a simulação tenha um efeito gerador conhecido, esse conhecimento não estaria
disponível em uma avaliação retrospectiva real.

**Registro do estudante:** qual variável pode causar simultaneamente seleção
para atendimento e melhora no escore? Em que direção sua omissão poderia
deslocar a estimativa? Que pedidos ficam fora do sistema de referência e como
isso limita a validade externa?

---

## 2. Dicionário de dados e quadro de variáveis

A unidade de observação é um pedido. Cada variável deve ser definida **antes**
de examinar os resultados. Os exemplos abaixo são valores possíveis em uma
base fictícia e não observações reais. As covariáveis de ajuste são aferidas
na entrada do pedido, antes da exposição, para reduzir o risco de controlar
uma consequência do próprio atendimento.

| Nome | Significado | Tipo | Escala | Papel | Exemplos |
|---|---|---|---|---|---|
| `id_pedido` | Código artificial único do pedido | Texto | Identificador, sem escala inferencial | Chave | `PED-0001`, `PED-0002` |
| `atendimento_programa` | Recebeu orientação jurídica do programa | Categórica binária | Nominal binária | Independente principal | `Sim`, `Não` |
| `pontuacao_acesso` | Escore construído de acesso efetivo, em pontos | Quantitativa contínua | Intervalar assumida | Dependente primária | `45,3`, `62,1` |
| `desfecho_administrativo` | Estado do pedido na janela de observação | Categórica multinomial | Nominal, cinco níveis | Dependente secundária | `deferido_integral`, `pendente` |
| `complexidade_previa` | Complexidade registrada na entrada | Categórica de três níveis | Ordinal | Confundidora observada | `1`, `2`, `3` |
| `vulnerabilidade_previa` | Indicador de vulnerabilidade antes do atendimento | Categórica binária | Nominal binária | Confundidora observada | `0`, `1` |
| `unidade` | Unidade administrativa de registro | Categórica multinomial | Nominal | Contexto e cluster | `U01`, `U30` |
| `ano_pedido` | Ano de protocolo do pedido | Temporal discreta | Calendário | Contexto temporal | `2023`, `2025` |
| `documentacao_inicial` | Documentos completos na entrada | Categórica binária | Nominal binária | Covariável pré-exposição | `Sim`, `Não` |
| `acesso_digital_nao_observado` | Facilidade de usar o portal, gerada apenas para ilustrar confundimento | Latente binária | Nominal binária | Confundidora oculta, excluída da base analítica | `0`, `1` no gerador |

**Regra de mensuração do escore.** O zero da pontuação será convencional e
diferenças iguais serão tratadas como comparáveis. Isso é uma **hipótese de
mensuração**; um escore produzido em pesquisa real exigirá descrição de itens,
pesos, consistência, estabilidade temporal e evidências de validade. A escala
não se torna intervalar apenas porque o código armazena valores decimais.

**Regra do desfecho.** Cada pedido terá uma única categoria entre
`deferido_integral`, `deferido_parcial`, `indeferido`, `pendente` e
`arquivado`, segundo uma janela de acompanhamento previamente delimitada.
Esses estados são tratados como **nominais**: sua ordem de exibição não
autoriza médias, distâncias numéricas nem interpretação ordinal.

**Para adaptar:** substitua cada linha por definição operacional, fonte,
momento de aferição, códigos admitidos, regra de ausências e ao menos dois
exemplos. Registre quem codificou as informações e como divergências de
classificação foram resolvidas.

In [ ]:
# Fixar os cinco rótulos nominais antes de simular o desfecho.
CATEGORIAS_DESFECHO = [
    "deferido_integral",
    "deferido_parcial",
    "indeferido",
    "pendente",
    "arquivado",
]

# Listar apenas variáveis observáveis que poderão entrar no ajuste.
COVARIAVEIS_PREVIAS = [
    "complexidade_previa",
    "vulnerabilidade_previa",
    "documentacao_inicial",
    "unidade",
    "ano_pedido",
]
VARIAVEIS_ANALITICAS = [
    "id_pedido",
    "atendimento_programa",
    "pontuacao_acesso",
    "desfecho_administrativo",
    *COVARIAVEIS_PREVIAS,
]

# Construir o mesmo quadro em pandas para validar o esquema em código.
registros_dicionario = [
    ("id_pedido", "Código artificial único do pedido", "Texto", "Identificador", "Chave", "PED-0001; PED-0002"),
    ("atendimento_programa", "Orientação jurídica recebida", "Categórica binária", "Nominal binária", "Independente principal", "Sim; Não"),
    ("pontuacao_acesso", "Escore construído de acesso efetivo", "Quantitativa contínua", "Intervalar assumida", "Dependente primária", "45,3; 62,1"),
    ("desfecho_administrativo", "Estado do pedido na janela definida", "Categórica multinomial", "Nominal, cinco níveis", "Dependente secundária", "deferido_integral; pendente"),
    ("complexidade_previa", "Complexidade registrada na entrada", "Categórica de três níveis", "Ordinal", "Confundidora observada", "1; 2; 3"),
    ("vulnerabilidade_previa", "Vulnerabilidade anterior ao atendimento", "Categórica binária", "Nominal binária", "Confundidora observada", "0; 1"),
    ("unidade", "Unidade administrativa de registro", "Categórica multinomial", "Nominal", "Contexto e cluster", "U01; U30"),
    ("ano_pedido", "Ano de protocolo do pedido", "Temporal discreta", "Calendário", "Contexto temporal", "2023; 2025"),
    ("documentacao_inicial", "Documentos completos na entrada", "Categórica binária", "Nominal binária", "Covariável pré-exposição", "Sim; Não"),
    ("acesso_digital_nao_observado", "Facilidade de uso do portal; apenas no gerador", "Latente binária", "Nominal binária", "Confundidora oculta", "0; 1 no gerador"),
]
dicionario = pd.DataFrame(
    registros_dicionario,
    columns=["nome", "significado", "tipo", "escala", "papel", "exemplos"],
)

# Confirmar unicidade, cobertura do esquema e preenchimento das definições.
assert dicionario["nome"].is_unique, "Há nomes de variáveis duplicados."
assert len(CATEGORIAS_DESFECHO) == 5, "O desfecho deve ter cinco níveis."
assert len(set(CATEGORIAS_DESFECHO)) == 5, "Os níveis devem ser distintos."
assert "pontuacao_acesso" in VARIAVEIS_ANALITICAS
assert "atendimento_programa" in VARIAVEIS_ANALITICAS
assert set(VARIAVEIS_ANALITICAS).issubset(set(dicionario["nome"]))
assert set(COVARIAVEIS_PREVIAS).issubset(set(VARIAVEIS_ANALITICAS))
assert "acesso_digital_nao_observado" not in VARIAVEIS_ANALITICAS
for coluna in dicionario.columns:
    assert dicionario[coluna].notna().all(), f"Campo vazio em {coluna}."
    assert dicionario[coluna].astype(str).str.strip().ne("").all(), (
        f"Campo sem conteúdo em {coluna}."
    )

# Exibir a versão verificável; a tabela Markdown acima permanece legível sem execução.
display(dicionario)
print("Variáveis analíticas documentadas:", len(VARIAVEIS_ANALITICAS))
print("Categorias nominais do desfecho:", ", ".join(CATEGORIAS_DESFECHO))

### Validade das medidas e papel das variáveis

O quadro separa o que foi observado do que é apenas parte do mecanismo
gerador. `acesso_digital_nao_observado` será omitida da base analítica para
ilustrar que um coeficiente ajustado ainda pode sofrer confundimento. Não se
deve afirmar que a variável foi medida no estudo observacional.

Documentação **inicial** pode ser um controle prévio; documentação corrigida
**após** o atendimento poderia ser mediadora. Controlar esta última mudaria a
pergunta estimada e poderia remover parte da associação de interesse. O mesmo
cuidado vale para qualquer variável registrada durante a tramitação.

**Registro do estudante:** qual é a evidência de que o escore mede acesso
efetivo? Duas pessoas poderiam codificar o mesmo desfecho da mesma forma?
Quais categorias estão sujeitas a erro ou censura pela janela retrospectiva?
Que variável oculta plausível alteraria a direção da associação observada?